# **02477 Bayesian Machine Learning | Comprehensive study notes**



---

###  **Table of Contents**


7. [Week 7 | multi-class, decision theory & calibration](#week7)

8. [Week 8 | monte carlo & metropolis-hastings](#week8)

9. [Week 9 | advanced MCMC & convergence diagnostics](#week9)

10. [Week 10 | variational inference & ELBO](#week10)

11. [Week 11 | Black-box variational inference (BBVI)](#week11)

12. [Week 12 | Bayesian neural networks](#week12)


---

<a id='week7'></a>

<div class="alert alert-block alert-info">

### **Week 7 | Multi-class classification, decision thery & calibration**

Gaussan processes and neural networks, generalized linear models and non-gaussian likelihoods, generalization, evaluation, decision theory, calibration

</div>

##### **7.0 Wide neural networks**

If we have a NN with single hidden layer and $H$ neurons and one output:
$$
z(x) = h(W_1 x + b_1)
$$
$$
f(x)=W_2z(x)+b_2
$$
If we assume activation function is bounded and $h(x)=\tanh(x)$ and $W_1$ and $b_1$ have i.i.d zero-mean Gaussian priors, and $W_2$ and $b_2$ have zero-mean and prior variances $\sigma^2_w$ and $\sigma^2_b$ respectively, with $\sigma^2_w = \frac{1}{H}$ as prior variance.

The mean and variance of $f(x)$ are then:
$$
\mathbb{E}[f(x)] = \mathbb{E}[W_2z(x)+b_2] = \sum_{i=1}^H \mathbb{E}[w_j] \mathbb{E}[h_j(x)] + \mathbb{E}[b_2]
$$
So the neural network is a zero-mean stochastic process.

<span style="color: red;">

The key result is as $H \to \infty$, $f(x)$ converges to a gaussian process (GP). This connects to NNs to GPs and motivates the GP as an infinite-width limit of NNs.

</span>

##### **7.1 Softmax for multi-class classification**


For $K$ classes with $f_k(x) = w_k^T \phi(x)$:

<span style="color: blue;">

$$p(y=k|x) = \text{softmax}_k(f(x)) = \frac{e^{f_k(x)}}{\sum_{j=1}^K e^{f_j(x)}}$$

</span>

<span style="color: red;">

Here, $f_k(x) = w_k^T \phi(x)$ is the linear predictor for class $k$, $K$ is number of classes, and output is a probability vector summing to $1$.

</span>


##### **7.2 Uncertainty decomposition**




For a categorical predictive distribution $\pi_k = p(y^*=k|y,x^*)$:

| Measure | Formula | Meaning |
|---------|---------|----------|
| **Confidence** | $C = \max_k \pi_k$ | How sure is the model? |
| **Total entropy** | $H = -\sum_k \pi_k \log \pi_k$ | Total uncertainty |
| **Aleatoric** | $\mathbb{E}_{p(w\|y)}[H(y^*\|w,x^*)]$| Inherent uncertainty |
| **Epistemic** | $H_{\text{total}} - H_{\text{aleatoric}}$ | Model uncertainty |


In aleatoric we have:

<span style="color: blue;">

$$
H(y^*|w,x^*) = -\sum_k p(y^*=k|w,x^*) \log p(y^*=k|w,x^*)
$$

</span>

Which is the entropy of the predictive distribution for a fixed $w$. We then average this over the posterior to get expected aleatoric uncertainty.

Max entropy = $\log K$ (uniform), achieved when model is maximally uncertain.


##### **7.2.1 Generalization error**

The generalization error (or expected loss, risk, out-of sample error) for a model $\hat{y}(x)$ is defined as:

<span style="color: blue;">

$$
\mathcal{R}_{\hat{y}} = \mathbb{E} [ \mathcal{L}(y, \hat{y}(x)) ] = \int \int \mathcal{L}(y, \hat{y}(x)) p(x,y) dx dy
$$

</span>

For i.i.d samples $(x_i^*, y_i^*) \sim p(x,y)$ is:

<span style="color: blue;">

$$
\mathcal{R}_{\hat{y}} = \hat{\mathcal{R}}_{\hat{y}}^\text{test} = \frac{1}{N_\text{test}} \sum_{i=1}^{N_\text{test}} \mathcal{L}(y_i^*, \hat{y}(x_i^*))
$$

</span>

The estimator is accurate when the test set is large, and then the empirical risk for the training set is:

<span style="color: blue;">

$$
\mathcal{R}_{\hat{y}} \approx \hat{\mathcal{R}}_{\hat{y}}^\text{train} = \frac{1}{N_\text{train}} \sum_{i=1}^{N_\text{train}} \mathcal{L}(y_i, \hat{y}(x_i))
$$

</span>

<span style="color: red;">

Here, $\mathcal{R}_{\hat{y}}$ is the true risk (generalization error), $\hat{\mathcal{R}}_{\hat{y}}^\text{test}$ is the empirical test risk and $\mathcal{L}(y, \hat{y}(x))$ is the loss function.

</span>

Many learning algorithms can be expressed as empirical risk minimization (ERM):
$$
\hat{w} = \argmin_w \hat{\mathcal{R}}_{\hat{y}}^\text{train}
$$
For ERM, the training error is optimistic meaning:
$$
\mathbb{E}[\hat{\mathcal{R}}_{\hat{y}}^\text{train}] \leq \mathbb{E}[\hat{\mathcal{R}}_{\hat{y}}^\text{test}]
$$
The generalization error with respect to squared loss for a linear regression model is:

<span style="color: blue;">

$$
\mathcal{R}_{\hat{w}} = (w-\hat{w})^2 + \sigma^2
$$

</span>

This is linear regression squared loss where $(w-\hat{w})^2$ is the bias term, and $\sigma^2$ is the irreducible aleatoric noise.

##### **7.3 Bayesian decision theory**


**Setup:**

> Actions: $\hat{y} \in \{1,\dots,K\}$

> Utility function: $U(y, \hat{y})$ = gain for predicting $\hat{y}$ when truth is $y$

> (Or equivalently, loss $L(y,\hat{y}) = -U(y,\hat{y})$)

**Optimal decision** = maximise expected utility:

$$\hat{y}^* = \arg\max_{\hat{y}} \mathbb{E}_{p(y^*|y,x^*)}[U(y^*, \hat{y})] = \arg\max_{\hat{y}} \sum_k \pi_k U(k, \hat{y})$$

**Special case 0/1 loss** (misclassification rate):

$$U(y,\hat{y}) = \mathbb{I}[y = \hat{y}] \Rightarrow \hat{y}^* = \arg\max_k \pi_k$$

So picking the most probable class is optimal under 0/1 loss.

<span style="color: red;">

**Reject option:** Abstain if $C = \max_k \pi_k < \theta_{\text{reject}}$, where $\theta_{\text{reject}} \in (0,1)$ is a threshold - if the model is not confident enough in any calss, then abstain rather than guess (abstain means to not make prediction).

</span>


##### **7.4 Decision theory for regression**

| Loss function | Optimal predictor |
|---|---|
| Squared loss $(y - \hat{y})^2$ | $\hat{y} = \mathbb{E}[y^*\|x^*]$ (posterior mean) |
| Absolute loss $\|y - \hat{y}\|$ | $\hat{y} = \text{median}(p(y^*\|x^*))$ |
| 0/1 loss | $\hat{y} = \text{mode}(p(y^*\|x^*))$ (MAP) |


##### **7.4.5 Decision theory for classification**

This example is for binary problems with 0/1 utility function given by $\mathcal{U}(y,\hat{y}(x)) = [y=\hat{y}(x)]$, so like:

| $\mathcal{U}(y^*, \hat{y}(x^*))$ | $\hat{y}(x^*)=0$ | $\hat{y}(x^*)=1$ |
|---|---|---|
| $y^*=0$ | 1 | 0 |
| $y^*=1$ | 0 | 1 |


And then if $p = p(y^*=1|y,x^*)$ denotes the posterior class probability for input point $x^*$ then:

<span style="color: blue;">

$$
\mathbb{E}[\mathcal{U}(y^*, \hat{y}(x^*))] = \sum_{y^*} p(y^*|y,x^*) \mathcal{U}(y^*, \hat{y}(x)) = (1-p)[0=\hat{y}(x)]+p[1=\hat{y}(x)]
$$

</span>

So for instance, if we choose $\hat{y}(x)=0$ then:
$$
\mathbb{E}[\mathcal{U}(y^*, \hat{y}(x^*))] = (1-p) \cdot 1 + p \cdot 0 = 1-p
$$
If we choose $\hat{y}(x)=1$ then:
$$
\mathbb{E}[\mathcal{U}(y^*, \hat{y}(x^*))] = (1-p) \cdot 0 + p \cdot 1 = p
$$
If we pick the class with largest poesterioer prediction probability, then it is Bayes optimal under the 0/1 loss function.

Furthermore, here is a formula for how certain we have to be before we dare to predict $\hat{y}^*=0$ given:


$$
\mathbb{E}[\mathcal{U}(y^*,0)]  \geq \mathbb{E}[\mathcal{U}(y^*,1)]
$$
$$
\Rightarrow (1-p) \mathcal{U}(0,0) + p \mathcal{U}(1,0) \geq (1-p) \mathcal{U}(0,1) + p \mathcal{U}(1,1)
$$

This is the threshold formula, which is presented as a blue function further below (its final form).


**THIS IS THE GENERAL RECIPE FOR DECISION THEORY NOT GIVEN A SPECIFIC UTILITY FUNCTION**

Given posterior class probability $p = p(y^*=1|y,x^*)$, and a utility matrix:

| | $\hat{y}=0$ | $\hat{y}=1$ |
|---|---|---|
| $y^*=0$ | $\mathcal{U}(0,0)$ | $\mathcal{U}(0,1)$ |
| $y^*=1$ | $\mathcal{U}(1,0)$ | $\mathcal{U}(1,1)$ |

**Step 1:** Compute expected utility for each decision:

<span style="color: blue;">

$$\mathbb{E}[\mathcal{U}(y^*, \hat{y}=0)] = (1-p)\mathcal{U}(0,0) + p\,\mathcal{U}(1,0)$$
$$\mathbb{E}[\mathcal{U}(y^*, \hat{y}=1)] = (1-p)\mathcal{U}(0,1) + p\,\mathcal{U}(1,1)$$

</span>

**Step 2:** Pick the decision with highest expected utility:
$$\hat{y}^* = \arg\max_{\hat{y} \in \{0,1\}} \mathbb{E}[\mathcal{U}(y^*, \hat{y})]$$


**Threshold formula** (predict $\hat{y}=0$ when):

<span style="color: blue;">

$$p \leq \frac{\mathcal{U}(0,0) - \mathcal{U}(0,1)}{\mathcal{U}(0,0) - \mathcal{U}(0,1) + \mathcal{U}(1,1) - \mathcal{U}(1,0)}$$

</span>

##### **7.5 Model calibration**


A model is **well-calibrated** if predicted probabilities match empirical frequencies:
$$P(\text{correct} | C = p) = p \quad \forall p$$

**Expected Calibration Error (ECE):**

<span style="color: blue;">

$$\text{ECE} = \sum_{b=1}^B \frac{|B_b|}{N}|\text{acc}(B_b) - \text{conf}(B_b)|$$

</span>

Visualised as a **reliability diagram** (calibration plot): confidence on x-axis, accuracy on y-axis with perfect calibration = diagonal.

**Overconfident:** curve below diagonal.  

**Underconfident:** curve above diagonal.

The average accuracy for a bin $b$ is:

<span style="color: blue;">

$$
\text{acc}(B_b) = \frac{1}{|B_b|} \sum_{i \in B_b} \mathbb{I}[\hat{y}_i = y_i]
$$

</span>

The average confidence for a bin $b$ is:

<span style="color: blue;">

$$
\text{conf}(B_b) = \frac{1}{|B_b|} \sum_{i \in B_b} C_i
$$

</span>

Where:
$$
\hat{y}_i = \argmax_c p(y^*_i=c|y,x^*_i)
$$
$$
C_i = \max_c p(y^*_i =c |y,x^*_i)
$$

Bins are divided such that:
$$
I_b = \left( \frac{b-1}{B}, \frac{b}{B} \right]
$$

##### **7.6 Generalized linear models (GLMs)**

The first component of a GLM is the linear model:
$$
f(x)=\phi(x)^Tw
$$
The second component is a link function $g$ that relates the mean of the linear model to the mean of the response variable $\mathbb{E}[y(x)|x] = \mu(x)$ so:
$$
g(\mu(x)) = f(x) \Rightarrow \mathbb{E}[y|x] = \mu(x) = g^{-1}(f(x))
$$
The third component is the distribution $p(y_n|x_n)$ for the response variable $y_n$, so like a poisson, binomial or gamma distribution.

##### **7.6.1 GLM likelihood selection - which model for which data?**


When a question changes the data type, change the likelihood and link function. This is the decision table:

| Observed $y_n$ | Likelihood | Link $g$ | Mean $\mu(x)$ | Keyword signals |
|---|---|---|---|---|
| Real-valued, additive noise | $\mathcal{N}(\mu, \sigma^2)$ | Identity | $w^T\phi(x)$ | "regression", "continuous", "noise $\epsilon$" |
| Binary $\{0,1\}$ | $\text{Bernoulli}(\mu)$ | Logit (sigmoid) | $\sigma(w^T\phi(x))$ | "binary", "classification", "probability of success" |
| Count $\{0,1,2,\ldots\}$ | $\text{Poisson}(\mu)$ | Log | $\exp(w^T\phi(x))$ | "count", "events", "non-negative integer" |
| Positive real $\mathbb{R}^+$ | $\text{Gamma}(\alpha,\beta)$ | Log | $\exp(w^T\phi(x))$ | "positive", "duration", "waiting time" |
| Multi-class $\{1,\ldots,K\}$ | $\text{Categorical}(\pi)$ | Softmax | $\text{softmax}(W\phi(x))$ | "classes", "categories", "$K$ outcomes" |

**The swap rule:** The linear model $f(x) = w^T\phi(x)$ stays the same. The Gaussian prior on $w$ stays the same. Only the likelihood $p(y_n \mid f(x_n))$ changes - everything upstream (prior, MAP, Laplace) follows the same procedure.

**Heteroscedastic extension:** If the *variance* also depends on the input, add a second output head:

$$p(y_n \mid w, x_n) = \mathcal{N}(y_n \mid f_1(x_n|w),\; e^{f_2(x_n|w)})$$

The log-variance parameterisation $e^{f_2}$ guarantees $\sigma^2 > 0$. The log-likelihood becomes:

$$\log p(y_n|w,x_n) = -\tfrac{1}{2}f_2(x_n|w) - \frac{(y_n - f_1(x_n|w))^2}{2\,e^{f_2(x_n|w)}} + \text{const}$$

**When to recognize this:** the question says "the variance depends on $x$", "input-dependent noise", or asks us to "extend the model to capture heteroscedastic noise."

##### **7.7 Generalized GP/NN models**

First component is where we replace the linear model with a GP (or a NN):
$$
f(x) \sim \mathcal{GP}(0, k(x,x'))
$$

Second component is the link function $f$ that related the mean of the linear model to the mean of the response variable $t(x)$:
$$
g[\mu(x)] = f(x) \Rightarrow \mathbb{E}[y|x] = \mu(x) = g^{-1}[f(x)]
$$

Third component is the distribution $p(y|x)$ for the response variable $y(x)$ for instance poisson, binomial or gamma distribution.

##### **7.8 Bayesian poisson regression**

**Step 1:** We assume linear model with $x = \begin{bmatrix} 1 & \text{age} \end{bmatrix}^T$ with:
$$
f(x) = w_0 + w_1 \cdot \text{age} \Rightarrow f(x)=x^T w
$$

**Step 2:** Since $\mu > 0$ so we use the log link function $\log (\mu) = f(x)$ aka:
$$
\mu(x) = \exp(f(x)) = \exp(x^T w)
$$

**Step 3:** We use a poisson likelihood for count data:
$$
\text{Poisson}(y_n=k|\mu) = \frac{\mu^k e^{-\mu}}{k!}
$$

**Step 4:** The likelilhood:
$$
y_n |x_n, w \sim \text{Poisson}(\mu_n)
$$

**Step 5:** Use a Gaussian prior for $w$:
$$
p(w) = \mathcal{N}(w|0, I)
$$

**Step 6:** Using $\mu_n = \mu(x) = \exp(x_n^T w)$ so the joint model becomes:
$$
p(y,w) = p(y|w)p(w) = \prod_{n=1}^N \text{Poisson}(y_n|\exp(x_n^T w)) \cdot \mathcal{N}(w|0, I)
$$

**Step 7:** Use Laplace approximation again because the posterior is intractable:
$$
p(w|y) = \frac{p(y|w)p(w)}{p(y)} \approx q(w) = \mathcal{N}(w|m, S)
$$

**Step 8:** The approximate posterior predictive distribution for a new point $x_*$ with $\mu_* = g^{-1}(x^T_* w)$ is:
$$
p(y_* =k |y) \approx \int p(y_*=k|w)q(w)dw = \int \text{Poisson}(y_* =k |\mu_*) \mathcal{N}(w|m, S)dw
$$

**Step 9:** Now calculate predictions, for example $\text{age}= 30$ so let $x_* = \begin{bmatrix} 1 & 30 \end{bmatrix}^T$ so $f_*=x_*^Tw$ with (numbers are from the slides):
$$
p(f_*|y) = \mathcal{N}(f_*|x_*^Tm, s_*TSx_*) \approx \mathcal{N}(f_*|0.8, 0.2^2)
$$

<span style="color: red;">

Numbers $0.8 = \mu_{f^*} = x_*^Tm$ and $0.2^2 = \sigma^2_{f^*} = s_*TSx_*$ are the mean and variance of the Gaussian distribution for $f_*$.

</span>

**Step 10:** We can now calculate the distributions of $\mu_*|y$ and $y_*|y$ using the samples from the Gaussian distribution above, where for each sample:
$$
\mu_*^{(s)} = \exp(f_*^{(s)})
$$
$$
y_*^{(s)} | \mu_*^{(s)} \sim \text{Poisson}(\mu_*^{(s)})
$$
**Step 11:** Finally, calculate the sample means and more (like variance or percentiles):
$$
\mathbb{E}[\mu_*^{(s)}|y] \approx \frac{1}{S} \sum_{s=1}^S \mu_*^{(s)} \approx 2.34
$$
$$
\mathbb{E}[y_*^{(s)}|y] \approx \frac{1}{S} \sum_{s=1}^S y_*^{(s)} \approx 2.34
$$

##### **7.8.1 Poisson log-posterior - the function we implement for MCMC/MAP**



> **When the exam asks us to run MCMC or find the MAP for a Poisson GLM**, we need
> the log-posterior. Here it is, fully written out.

For the model $y_n | x_n, w \sim \text{Poisson}(\mu_n)$, $\mu_n = \exp(f(x_n))$,
$f(x_n) = w^T x_n$ (or any other linear combination), with prior $p(w) = \mathcal{N}(w \mid 0, \alpha^{-1} I)$:

**Log-likelihood** (sum over all $N$ observations):
$$\log p(y \mid w) = \sum_{n=1}^N \left[ y_n \cdot w^T x_n - \exp(w^T x_n) - \log(y_n!) \right]$$

> The $\log(y_n!)$ term is a constant wrt. $w$ - drop it when optimizing or using as MCMC target.

**Log-prior**:
$$\log p(w) = -\frac{\alpha}{2} w^T w + \text{const}$$

**Log-posterior** (what we pass to Metropolis or use for MAP):
$$\log p(w \mid y) \propto \sum_{n=1}^N \left[ y_n \cdot w^T x_n - \exp(w^T x_n) \right] - \frac{\alpha}{2} w^T w$$

```python

```python
def log_posterior_poisson(w, x, y, alpha=8.0):
    """
    w     : array of shape (D,)
    x     : array of shape (N, D)
    y     : array of shape (N,)  — count observations
    alpha : prior precision (prior is N(0, alpha^{-1} I))
    """
    f = x @ w                          # shape (N,)  — linear predictor
    log_lik = jnp.sum(y * f - jnp.exp(f))   # drop log(y!) constant
    log_prior = -0.5 * alpha * jnp.dot(w, w)
    return log_lik + log_prior
```

**Prior mean of $\mu(x^*)$** - a common exam sub-question:
$$\mathbb{E}_{p(w)}[\mu(x^*)] = \mathbb{E}_{p(w)}[\exp(w^T x^*)]$$

Since $w^T x^* \sim \mathcal{N}(0, \|x^*\|^2 / \alpha)$ under the prior, this is the
moment-generating function of a Gaussian:
$$\mathbb{E}[\exp(w^T x^*)] = \exp\!\left(\frac{\|x^*\|^2}{2\alpha}\right)$$

> **2024R Q4.2 pattern:** $\mu(x_n) = \exp(3 + w_1 x_n + w_2 x_n^2)$, $x^* = 0$.
> Under prior $w \sim \mathcal{N}(0, \alpha^{-1}I)$: $f(x^*) = 3 + 0 + 0 = 3$ deterministically.
> So $\mathbb{E}[\mu(x^*)] = \exp(3) \approx 20.09$.
> **Rule:** if $x^* = 0$ zeroes out all $w$-dependent terms, $\mu(x^*) = \exp(\text{constant})$ is not random.

**Posterior predictive - the two-step sampling procedure:**
```python
# Step 1: sample f* from its posterior marginal
f_star_samples = x_star @ m + jnp.sqrt(x_star @ S @ x_star) * random.normal(key, (S,))

# Step 2: compute mu* then sample y* from Poisson
mu_star_samples = jnp.exp(f_star_samples)
y_star_samples = jnp.array(np.random.poisson(np.array(mu_star_samples)))  # numpy, not JAX!

# Credibility interval
jnp.percentile(y_star_samples, jnp.array([5.0, 95.0]))  # 90% CI
```

<span style="color: red;">

**Critical trap:** `mu_star` is the Poisson *rate*, not $y^*$ itself. Always sample
$y^* \sim \text{Poisson}(\mu^*)$ as a second step. Use `np.random.poisson` (numpy), not JAX.

</span>

##### **7.9 GP models adaption to likelihoods**

First, laplace approximation for GP classifaction:
$$
p(f|y) = \frac{p(y|f)p(f)}{p(y)} \approx q(f) = \mathcal{N}(f|m, S)
$$
The log joint of the target $y$ and latent function values $f$ is:

<span style="color: blue;">

$$
\log p(y,f) = \log p(y|f)+ \log p(f) 
$$
$$
= \sum_{n=1}^N \log p(y_n |f_n)- \frac{N}{2} - \frac{N}{2} \log (2 \pi) - \frac{1}{2} |K| - \frac{1}{2} f^T K^{-1} f
$$

</span>

The gradient and Hesisan of the log joint is:

<span style="color: blue;">

$$
\nabla_f \log p(y,f) = \sum_{n=1}^N \nabla_{f_n} \log p(y_n |f_n) - K^{-1} f
$$
$$
\nabla^2_f \log p(y,f) = \sum_{n=1}^N \nabla^2_{f_n} \log p(y_n |f_n) - K^{-1}
$$

</span>

<span style="color: red;">

Here, $\log p(g_n | f_n)$ is the log-likelihood for a single observation (depends on chosen likelihood - bernoulli for classification, poisson for count data), and $-\frac12 f^TK^{-1}f$ is the log-prior on $f$ under the GP.

</span>

<a id='week8'></a>

<div class="alert alert-block alert-info">

### **Week 8 | monte carlo & metropolis-hasting**

Calibration, monte carlo methods, simple sampling methods, markov chain monte carlo methods

</div>

##### **8.1 Why we need sampling**



For most models, the posterior $p(\theta|y)$ is **analytically intractable**. MCMC methods generate (correlated) samples from $p(\theta|y)$ without knowing the normalisation constant, so as the slides say, they break the curse of dimensionality.


##### **8.1.5 Rejection and importance sampling**

Rejection sampling is for generating samples from $p(z)$ with:
$$
p(z) = \frac{1}{Z} \tilde{p}(z)
$$
We let $q(z)$ be an easy distribution and $K>0$ a constant such that $\tilde{p}(z) \leq Kq(z)$ for all $z$, then the steps in the sampling procedure are:

**Step 1:** Sample $z \sim q(z)$.

**Step 2:** Sample $u|z$ from a uniform distribution $u \sim \mathcal{U}[0, Kq(z)]$.

**Step 3:** If $u > \tilde{p}(z)$ then reject $z$ and return to step 1, otherwise accept $z$.

<span style="color: red;">

Here, $q(z)$ is the proposal distribution (easy to sample from), $p(z)$ is the target distribution (hard to sample from directly), and $w_i = p(z^i)/q(z^i)$ are the importance weights for the mismatch between $q$ and $p$.

</span>

Next, **importance sampling** is a technique for estimating the expectaiton of a distribution $p(z)$ when we cannot sample from $p$ directly. We let $q$ be an easy distribution again then:

<span style="color: blue;">

$$
\mathbb{E}[f(z)] = \int f(z) p(z) dz =\mathbb{E}_q \left[f(z) \frac{p(z)}{q(z)}\right]
$$
$$
\approx \frac{1}{S} \sum_{i=1}^S f(z^{i})\frac{p(z^i)}{q(z^i)} = \hat{f},\quad  \text{for} \quad z^i \sim q(z)
$$

</span>

The ratios:

<span style="color: blue;">

$$
w_i = \frac{p(z^i)}{q(z^i)}
$$

</span>

They are called the importance weights, and the variance of the estimator is:
$$
\hat{\sigma}^2_q = \frac{1}{S} \sum_{i=1}^S (w_i f(z^i) - \hat{f})^2
$$

##### **8.2 Monte carlo integration**


Goal:

Estimate:

<span style="color: blue;">

$$\bar{f} = \mathbb{E}_p[f(\theta)] = \int f(\theta)p(\theta)d\theta$$

</span>

With i.i.d. samples $\theta^{(1)}, \dots, \theta^{(S)} \sim p(\theta)$ so:

<span style="color: blue;">

$$\hat{f} = \frac{1}{S}\sum_{s=1}^S f(\theta^{(s)})$$

</span>

**Properties:**

Unbiased ($\mathbb{E}[\hat{f}] = \bar{f}$), variance $\mathbb{V}[\hat{f}] = \frac{1}{S}\mathbb{V}[f(\theta)]$ decreases as $1/S$.


**Common exam patterns - everything is just a sample mean:**

| What you want | $f(\theta)$ | Python |
|---|---|---|
| Posterior mean | $f(\theta) = \theta$ | `jnp.mean(samples)` |
| Posterior variance | $f(\theta) = (\theta - \mathbb{E}[\theta])^2$ | `jnp.var(samples)` |
| Probability $p(\theta > c)$ | $f(\theta) = \mathbb{I}[\theta > c]$ | `jnp.mean(samples > c)` |
| Probability $p(\theta < c)$ | $f(\theta) = \mathbb{I}[\theta < c]$ | `jnp.mean(samples < c)` |
| Any expectation $\mathbb{E}[g(\theta)]$ | $f(\theta) = g(\theta)$ | `jnp.mean(g(samples))` |
| Posterior predictive | $f(\theta) = p(y^*\|\theta, x^*)$ | `jnp.mean(likelihood(samples))` |


Here, note that the indicator function $\mathbb{I}[\theta > \tau]$ is 1 if $\theta > \tau$ and 0 otherwise, so the expectation of this function gives us the probability that $\theta$ exceeds the threshold $\tau$.



**Examples:**
```python
# E[(z-x)^2] from 2025 exam Q5.2
jnp.mean((samples[:,0] - samples[:,1])**2)

# p(x < 1) from 2025 exam Q5.3
jnp.mean(samples[:,1] < 1)

# p(z1 > z2) from 2024 exam
jnp.mean(samples[:,0] > samples[:,1])
```

> **Rule:** whatever the exam asks to estimate, just write it as a function of your samples and take `jnp.mean(...)`. Always discard warmup first.


##### **8.2.5 Markov chain monte carlo methods**

<!-- MCMC provides a way to sample from almost any distribution $p(w)$. Below is an example on a first-order markov chain, which is defined as a series of random variables with the following conditional independence property:
$$
p(z^{m+1}|z^1, \dots, z^m) = p(z^{m+1}|z^m)
$$
Implications of the markov assumption:
$$
p(z_1,z_2,z_3,z_4) = p(z_4|z_1,z_2,z_3)p(z_3|z_1,z_2)p(z_2|z_1)p(z_1) = p(z_4|z_3)p(z_3|z_2)p(z_2|z_1)p(z_1)
$$
The transitional kernel is defined as:
$$
T_m (z^m, z^{m+1}) = p(z^{m+1}|z^m)
$$
Where $T_m$ is homogenous if it is the same for all $m$.

The distribution of $z^{m+1}$ is found via the sum rule:
$$
p(z^{m+1}) = \int p(z^{m+1}|z^m)p(z^m)dz^m 
$$

Furthermore, a distribution is said to be invariant or stationary with respect to the markov chain, if each step does not change the dstribution.

> **The central idea:** Design a markov chain such that the target distribution $p(z)$ is invariant, then run the chain for a long time and collect samples from the chain, which will be approximately distributed according to $p(z)$. -->

The MH algorithm is guaranteed to converge to the target $p(\theta|y)$ as the stationary distribution, provided the chain is ergodic (irreducible and aperiodic). In practice this means: run long enough, do not get stuck.

(open this markdown for more information)

##### **8.3 The metropolis-hastings algorithm**


**Key insight:** We only need to evaluate $p(\theta|y)$ up to a constant, because $p(y)$ cancels:

<span style="color: blue;">

$$A_k = \min\!\left(1, \frac{p(\theta^*|y)\,q(\theta^{k-1}|\theta^*)}{p(\theta^{k-1}|y)\,q(\theta^*|\theta^{k-1})}\right) = \min\!\left(1, \frac{p(y,\theta^*)\,q(\theta^{k-1}|\theta^*)}{p(y,\theta^{k-1})\,q(\theta^*|\theta^{k-1})}\right)$$

</span>

The normalising constant $p(y)$ cancels!


**For symmetric proposals** $q(a|b) = q(b|a)$ (e.g. Gaussian) simplifies to Metropolis:

$$A_k = \min\!\left(1, \frac{p(\theta^*|y)}{p(\theta^{k-1}|y)}\right)$$

This is simplified because it assumes symmetric proposal dist $q(a|b) = q(b|a)$, so the proposal terms cancel out in the acceptance ratio.



**Algorithm:**

Start from initial value $\theta^0$.

Repeat for $k=1$ to $K$:

1. Given the last value $\theta^{k-1}$, generate candidate sample using proposal distribution:

<span style="color: blue;">

$$
\theta^* \sim q(\theta^*|\theta^{k-1})
$$

</span>

2. Compute acceptance probability $A_k$:

<span style="color: blue;">

$$
A_K = \min \left( 1, \frac{p(\theta^*)}{p(\theta^{k-1})}\right)
$$

$$
A_k = \min(1, \exp[\log p(\theta^*) - \log p(\theta^{k-1})])
$$

</span>

3. Simulate $u_k \sim \mathcal{U}[0,1]$ and define $\theta^k$ as:

<span style="color: blue;">

$$
\theta^k = \begin{cases} \theta^* & \text{if } u_k < A_k \\ \theta^{k-1} & \text{otherwise} \end{cases}
$$

</span>

4. Discard warm up samples if asked for it like `wamrup = 0.10 * samples` then put `samples = samples[int(warmup):]`


##### **8.3.5 Transition kernel for metropolis-hastings**

The transition kernel for Metropolis-Hastings is:
$$
T(\theta'|\theta) = \begin{cases} q(\theta'| \theta) A(\theta, \theta') & \text{if } \theta' \neq \theta \\ q(\theta|\theta)A(\theta, \theta) + \int q(\theta''|\theta)(1-A(\theta'' | \theta))d\theta'' & \text{if } \theta' = \theta \end{cases}
$$

**Step 1:** We set $\theta^0$ to som intial value (initialisation phase).

**Step 2:** We iterate $\theta^{k+1} | \theta^k \sim T(\theta^{k+1}|\theta^k)$ (warm-up phase).

**Step 3:** Eventually the distribution of $\theta^k$ will converge to the stationary distribution $p^*$ (sampling phase).

##### **8.4 CRITICAL what "use Gaussian proposal" means on the exam**


(GP classification, logistic regression, Poisson regression - any model where the posterior is intractable.)


This is always confusing so read carefully.

**The proposal distribution $q(\theta^*|\theta^{k-1}) = \mathcal{N}(\theta^*|\theta^{k-1}, \tau^2 I)$ is ALREADY built into the `metropolis` function:**

```python
proposal_dist = random.normal(key_proposal, shape=(num_params,)) * tau
theta_star = thetas[-1] + proposal_dist  # = theta^{k-1} + N(0, tau^2 I)
```

So when the exam says "use isotropic Gaussian proposal with std $\tau$", we just pass `tau=...` to the function. We never construct the proposal ourselves.

**What WE have to provide is `log_target` aka the log of the unnormalised posterior:**

$$\texttt{log\_target}(\theta) = \log p(y|\theta) + \log p(\theta) = \log\text{likelihood} + \log\text{prior}$$

**How to construct `log_target` from the model:**

Given a model, read off the log-likelihood and log-prior and add them:

```python
log_npdf = lambda x, m, v: -0.5*jnp.log(2*jnp.pi*v) - (x-m)**2/(2*v)

# Example: 2025 exam Part 5
# Model: x|z ~ N(0, exp(z)),  z ~ N(1, 1)
def log_target(theta):
    z, x = theta          # unpack parameters in order stored in theta
    log_lik   = log_npdf(x, 0, jnp.exp(z))   # p(x|z)
    log_prior = log_npdf(z, 1, 1)             # p(z)
    return log_lik + log_prior

# Example: 2025R exam Part 5 (neural network regression)
# Model: y|w,x ~ N(w2*tanh(w1*x), sigma2),  w ~ N(0, I)
def log_target(w):
    f = w[1] * jnp.tanh(w[0] * x1)
    log_lik   = log_npdf(y1, f, sigma2)
    log_prior = log_npdf(w[0], 0, 1) + log_npdf(w[1], 0, 1)
    return log_lik + log_prior
```

**General recipe for any exam question:**

1. Read the model definition from the question
2. Write `log_target` = log-likelihood + log-prior
3. Call `metropolis(log_target, num_params=..., tau=..., num_iter=10**4, theta_init=..., seed=0)`
4. Discard warmup: `samples = samples[int(0.1*len(samples)):]`
5. Estimate anything with `jnp.mean(f(samples))`

> **Note on tau:** tau is a tuning parameter, not derived from the model. The exam usually tells us what value to use. Rule of thumb: aim for ~20-50% acceptance rate. If not specified, `tau=1` is a safe default.

> **Note on parameter ordering:** when `theta` is a vector, we decide the order. Just be consistent like if we write `z, x = theta` when constructing `log_target`, then `samples[:,0]` is $z$ and `samples[:,1]` is $x$.





##### **8.5 Practical considerations**

| Issue | Solution |
|---|---|
| Burn-in / warm-up | Discard first 10-50% of samples |
| **Proposal variance $\tau$** | **Too small → slow mixing; too large → low acceptance. Aim for ~20-50%** |
| Multiple chains | Run from different `theta_init`, check traces look similar |
| Correlated samples | MCMC samples are correlated → effective sample size < $S$ |
| Check convergence | Traces should look like random noise (not trending or stuck) |

<a id='week9'></a>

<div class="alert alert-block alert-info">

### **Week 9**

MCMC theory, Hamiltonian Monte Carlo, the ULA/MALA algorithms, MCMC convergence diagnostics

</div>

##### **9.1 MCMC theory**


A Markov chain has a **stationary distribution** $p^*(\theta)$ if:

<span style="color: blue;">

$$p^*(\theta) = \int T(\theta|\theta') p^*(\theta') d\theta'$$

</span>

So it is required that $p^*(\theta)$ is a limiting distribution of the chain (so independent of the initial distribution). For a chain to converge to $p^*(\theta)$, it must be:

> **(A1) Irreducible:** All states are reachable

> **(A2) Aperiodic:** No deterministic cycles

> **(A3) Positive recurrent:** Returns to any state with positive probability

**Detailed balance** (sufficient for $p^*$ to be stationary), so if a chain satisfies this detailed balance condtion, then $p^*$ is a stationary distribution of the chain:

$$T(\theta'|\theta)\,p^*(\theta) = T(\theta|\theta')\,p^*(\theta')$$

MH with reasonable proposals satisfies detailed balance. Furthermore a chain that satisfies (A1)-(A3) is ergodic, meaning that the distribution of $\theta^k$ converges to $p^*$ as $k \to \infty$ aka:

<span style="color: blue;">

$$
p(\theta^k) \xrightarrow[k \to \infty]{} p^*(\theta)
$$

</span>


The MH algorithm is specifically designed to satisfy balance with target $p^*(\theta) = p(\theta | y)$, which is why it works without knowing the normalisation constant $p(y)$.

##### **9.1.5 Metropolized random walk**

For metropolis hasting algorithm with Gaussian proposal:
$$
q(\theta^*| \theta^k) = \mathcal{N}(\theta^*| \theta^k, \tau^2 I) \Leftrightarrow \theta^* = \theta^k + \tau \epsilon, \quad \epsilon \sim \mathcal{N}(0, I)
$$
This is called a **metropolized random walk** (MRW), but it can suffer from slow convergence/ mixing. The MRW proposal distribution is independent of the target distribution.

##### **9.2 Hamiltonian monte carlo (HMC)**


HMC is a special case of the Metropolis-Hastigs algorithm.

**Motivation:** Random walk MH has slow mixing in high-dimensional spaces. HMC uses **gradient information** to make directed proposals.

**Augmented system:** Introduce momentum $\nu \sim \mathcal{N}(0,I)$, define Hamiltonian:

<span style="color: blue;">

$$H(\theta, \nu) = E(\theta) + K(\nu) + \text{const}$$
$$ = \underbrace{-\log p(\theta|y)}_{\text{potential energy}} + \underbrace{\frac{1}{2}\nu^T\nu}_{\text{kinetic energy}} + \text{const} $$

</span>

**Leapfrog integration** (numerically solve Hamiltonian dynamics) which is when given a step-size $\eta>0$ and a state $(\theta_k, \nu_k)$ at time $t=k\eta$ we can compute $(\theta_{k+1}, \nu_{k+1})$ at time $t=(k+1)\eta$ using the following steps:

Half step for momentum $\nu$:

<span style="color: blue;">

$$\nu_{k+1/2} = \nu_k - \frac{\eta}{2}\nabla_\theta E(\theta_k)$$

</span>

Full-step for position $\theta$:

<span style="color: blue;">

$$\theta_{k+1} = \theta_k + \eta \nu_{k+1/2}$$

</span>

Half-step for momentum $\nu$:

<span style="color: blue;">

$$\nu_{k+1} = \nu_{k+1/2} - \frac{\eta}{2}\nabla_\theta E(\theta_{k+1})$$

</span>

<!-- $$\nu_{t+\eta/2} = \nu_t + \frac{\eta}{2}\nabla_\theta \log p(\theta_t|y)$$
$$\theta_{t+\eta} = \theta_t + \eta\, \nu_{t+\eta/2}$$
$$\nu_{t+\eta} = \nu_{t+\eta/2} + \frac{\eta}{2}\nabla_\theta \log p(\theta_{t+\eta}|y)$$ -->

**HMC algorithm:** repeat for $L$ leapfrog steps, then MH accept/reject.

> **Why HMC is better:** Proposals follow the curvature of the posterior, so they have much higher acceptance rates and can traverse the space in fewer steps.


<span style="color: red;">

Here, $\theta$ is the position (model parameters), $\nu$ is the momentum (auxiliary variable), $E(\theta) = -\log p(\theta|y)$ is the potential energy, and $K(\nu) = \frac{1}{2}\nu^T\nu$ is the kinetic energy.

</span>


The key insight is that $p(\theta, \nu) \propto \exp(-H(\theta, \nu)) = p(\theta|y) \cdot \mathcal{N}(\nu|0,I)$, so marginalizing out $\nu$ recovers the target distribution $p(\theta|y)$.

The algorithm is formally given as:

For $k=1,2, \dots, K$ do:

1. Sample **new momentum** variable $\nu_{k-1} \sim \mathcal{N}(0, I)$.

2. Use **leapfrog integration** to compute $(\theta^*, \nu^*) = \text{leapfrog}_{\eta, L}(\theta^{k-1}, \nu_{k-1})$.

3. Compute **acceptance probability* $a_k = \min (1, \exp[-H(\theta^*, \nu^*) + H(\theta^{k-1}, \nu_{k-1})])$.

4. Set $\theta_k = \theta^*$ with probability $a_k$, otherwise $\theta_k = \theta^{k-1}$.

Note also that if the step-size $\eta$ is sufficiently small such that $H(\theta^*, \nu^*) \approx H(\theta^{k-1}, \nu_{k-1})$ then the acceptance probability $a_k$ will be close to 1, so we can get very high acceptance rates with HMC.

We also have $\nabla_\theta E(\theta) = -\nabla_\theta \log p(\theta|y) = -\nabla_\theta \log p(y|\theta) - \nabla_\theta \log p(\theta)$, which is the negative gradient of log-joint. In JAX this is computed via `jax.grad`.

##### **9.2.5 HMC terminology summary (same as above, but collected)**


<span style="color: blue;">

We let $p_t(\theta)$ be the target distribution for parameters of interest, and refer to $\theta$ as the position of some imaginative particle.

We let $\nu \in \mathbb{R}^D$ be the **momentum** of the particle and assume $p(\nu) = \mathcal{N}(\nu|0, I)$.

We let $E(\theta) = - \log p_t(\theta) + \text{const}$ denote the **potential energy**.

We let $K(\eta) = - \log p(\nu) + \text{const}$ denote the **kinetic energy**.


</span>


##### **9.3 Convergence diagnostics**


$\hat{R}$ statistic (Potential Scale Reduction Factor)

Run $M$ chains, each of length $S$. Let $B$ = between-chain variance, $W$ = within-chain variance:

<span style="color: blue;">

$$\hat{R}^2 = \frac{S-1}{S} + \frac{1}{S}\frac{B}{W}$$

</span>

> $\hat{R} = 1$: chains have mixed perfectly so $B=W$. 

> $\hat{R} > 1$: chains have not converged with $B>W$.

> **Rule of thumb:** $\hat{R} < 1.1$ (or $< 1.01$ for critical applications) is when chains have mixed.

Effective Sample Size (ESS)

MCMC samples are correlated - $S$ correlated samples carry less information than $S$ i.i.d. samples:

<span style="color: blue;">

$$S_{\text{eff}} = \frac{S}{1 + 2\sum_{t=1}^\infty \rho_t}$$

</span>

Where $\rho_t$ is the autocorrelation at lag $t$.

Monte Carlo Standard Error (MCSE) which is the expected difference between the estimate $\hat{f}$ and the true value $\bar{f}$:

<span style="color: blue;">


$$\text{MCSE} = \frac{1}{\sqrt{S}} \sqrt{\mathbb{V}[f(w)]} = \frac{1}{\sqrt{S_{\text{eff}}}} \hat{\text{sd}} (f(w))$$

</span>

The correlation between two samples $w^i$ and $w^{i+t}$ is given by the autocorrelation function, and MCMC methods produces highly correlated samples:
$$
\rho_t = \frac{1}{\sigma^2} \int (w^i- \mu)(w^{i+t} - \mu) p(w) dw
$$
<span style="color: red;">

Here, $S$ is the chain length, $B= \frac{S}{M-1} \sum_m (\bar{\theta}_m - \bar{\theta})^2$ is the between-chain variance, and $W = \frac{1}{M} \sum_m \frac{1}{S-1} \sum_m s^2_m$ is within chain variance with $s_m^2$ being the sample variance of chain $m$.

</span>

##### **9.4 Pros and cons of MCMC methods**


| Method | Pros | Cons |
|--------|------|------|
| **MH** | Easy to implement, strong guarantees | Slow mixing, acceptance rate issues |
| **HMC** | Uses gradients → efficient, high-D friendly | Need gradient; $\eta$, $L$ tuning required |
| **NUTS** | Adaptive HMC, no tuning needed | Complex to implement |
| **Gibbs** | No acceptance step | Requires conditional distributions |

##### **9.5 The MALA algorithm**

If we do a single leapfrog step $L=1$:
$$
\theta_{k+1}^\text{MALA} = \theta_k + \eta \nu_{k+1/2}
$$
$$
= \theta_k + \frac{\eta^2}{2} \nabla_\theta \log p_t(\theta_k) + \eta \nu_k, \quad \nu_k \sim \mathcal{N}(0, I)
$$
Then the metropolis-adjusted langevin algorithm (MALA) is:
$$
q_\text{MALA}(\theta^*| \theta_k) = \mathcal{N}(\theta^* | \theta_k + \frac{\eta^2}{2} \nabla_\theta \log p_t(\theta_k), \eta^2 I)
$$
In MALA, the gradient can be estimated using mini-bacthing, but the acceptance probability still requires a full pass through the dataset.

The unadjusted langevin algorithm (ULA) is when we just take the step without the MH accept/reject, and it is faster but also biased in the sense that it willnot converge to the true target distribution.

> Note that $-\nabla E(\theta) = + \nabla \log p_t(\theta)$ but both forms appear throughout the course.

##### **9.6 MCMC methods overview**

| **Algorithm** | **Proposal** | **MH Step?** | **Properties** |
|---|---|---|---|
| **MRW** | $\boldsymbol{\theta}^* = \boldsymbol{\theta}_k + \eta\boldsymbol{\epsilon}$, $\quad \boldsymbol{\epsilon} \sim \mathcal{N}(0, I)$ | Yes | Random walk. Can suffer from slow exploration. |
| **ULA** | $\boldsymbol{\theta}_{k+1} = \boldsymbol{\theta}_k - \frac{\eta^2}{2}\nabla E(\boldsymbol{\theta}_k) + \eta\boldsymbol{\epsilon}$, $\quad \boldsymbol{\epsilon} \sim \mathcal{N}(0, I)$ | No | Unadjusted. Follows the gradient for directed exploration. Always accepts, resulting in a *biased* stationary distribution. |
| **MALA** | $\boldsymbol{\theta}^* = \boldsymbol{\theta}_k - \frac{\eta^2}{2}\nabla E(\boldsymbol{\theta}_k) + \eta\boldsymbol{\epsilon}$, $\quad \boldsymbol{\epsilon} \sim \mathcal{N}(0, I)$ | Yes | Adjusted with MH correction. Eliminates ULA's bias. Equivalent to HMC with $L=1$. Scales better than MRW. |
| **HMC** | $(\boldsymbol{\theta}^*, \boldsymbol{\nu}^*) = \text{Leapfrog}_{\eta,L}(\boldsymbol{\theta}_k, \boldsymbol{\nu}_k)$, $\quad \boldsymbol{\nu}_k \sim \mathcal{N}(\mathbf{0}, I)$ | Yes | Simulates Hamiltonian dynamics. Suppresses random walks, allowing for distant proposals with high acceptance rates. |

<a id='week10'></a>

<div class="alert alert-block alert-info">

### **Week 10 | Variational Inference & Mixture Models**

Variational inference, ELBO, KL divergence, mean-field approximation, CAVI, Gaussian mixture models, Dirichlet distribution
</div>

##### **10.0 HMC algorithms in a "nutshell"**

**Goal:** Generating samples from a target distribution $p(\theta) = \frac{1}{Z} \tilde{p}(\theta)$.

Recipe for generating the $k+1$'the sample using HMC is then:

1. Initialize $\theta_0' = \theta_k$ and $\nu_0' \sim \mathcal{N}(0, I)$.

2. For $\ell = 1, \dots, L$ do:
$$
\nu_{\ell}' = \nu_{\ell-1}' + \eta \nabla \log \tilde{p}(\theta_{\ell-1}')
$$
$$
\theta_{\ell}' = \theta_{\ell-1}' + \eta \Sigma^{-1} \nu_{\ell}'
$$

3. Set $\theta^* = \theta_L'$ and $\nu^* = \nu_L'$.

4. Compute acceptance probability:
$$
A_{k+1} = \min \left( 1, \frac{p(\theta^*, \nu^*)}{p(\theta_k, \nu_0')} \right)
$$

5. Appect proposal $\theta^*$ with probability $A_{k+1}$, otherwise set $\theta_{k+1} = \theta_k$.

(THIS IS THE SIMPLIFIED VERSION OF LEAPFROG, SEE SECTION 92 FOR THE FULLY CORRECT VERSION TO REFERENCE).

##### **10.1 Why variational inference?**


When the posterior $p(\theta|y)$ is intractable (non-conjugate, or too expensive for MCMC), we search for the **closest tractable approximation** within a **variational family** $\mathcal{Q}$:

<span style="color: blue;">

$$q^* = \arg\min_{q \in \mathcal{Q}} \text{KL}[q || p] = \argmax_{q \in \mathcal{Q}} \mathcal{L}[q]$$

</span>

Usually we have $\mathcal{D}[q||p]$, but here we already choose the $\text{KL}$ as divergence measure.

**The fundamental identity** (always true):

<span style="color: blue;">


$$\underbrace{\ln p(D)}_{\text{const}} = \underbrace{\mathcal{L}[q]}_{\text{ELBO}} + \underbrace{\text{KL}[q \| p]}_{\geq 0} \geq \mathcal{L}[q]$$

</span>

<span style="color: red;">

Where $\ln p(D)$ is the log evidence (constant w.r.t. $q$), $\mathcal{L}[q]$ is the 
**ELBO** (Evidence Lower BOund), and $\text{KL}[q\|p] \geq 0$ is the KL divergence 
between the approximation $q$ and the true posterior $p(\theta|y)$.

</span>

**Key insights:**

1. Maximizing $\mathcal{L}[q]$ is equivalent to minimizing $\text{KL}[q \| p]$ since $\text{KL}[q||p] \geq 0$ and $\ln p(D)$ is constant.

2. We only need to be able to evaluate the log joint distribution $p(z, D)$ and not the posterior $p(z|D)$, which is often tractable.

3. The ELBO $\mathcal{L}[q]$ is a lower bound on the log evidence $\ln p(D)$, so maximizing the ELBO also gives us a lower bound on the evidence since $\mathcal{L}[q] \leq \ln p(D)$.

**ELBO definition:**

<span style="color: blue;">

$$\boxed{\mathcal{L}[q] = \mathbb{E}_q[\ln p(D, z)] - \mathbb{E}_q[\ln q(z)]}$$
$$= \underbrace{\mathbb{E}_q[\ln p(D, \theta)]}_{\text{expected log-joint}} + \underbrace{H[q]}_{\text{entropy of } q}$$

</span>

**Alternative form** (very useful for exam calculations):

<span style="color: blue;">

$$\mathcal{L}[q] = \underbrace{\mathbb{E}_q[\ln p(y|\theta)]}_{\text{expected log-likelihood}} - \underbrace{\text{KL}[q(\theta) \| p(\theta)]}_{\text{KL to prior}}$$

</span>

##### **10.1.5 Variational family**

The variational family $\mathcal{Q}$ defines the collection of all possible approximations $q\in \mathcal{Q}$ we are willing to consider. Common choices include:

1. Full-rank Gaussian $q(z) = \mathcal{N}(z|\mu, \Sigma)$

2. Mean-field Gaussians $q(z) = \prod_{i=1}^D \mathcal{N}(z_i|\mu_i, \sigma^2_i)$

3. Factorized mean-field approximations $q(z) = \prod_{i=1}^D q_i(z_i)$ where $z = [z_1, z_2,  \dots, z_D]$ is partitioned into $D$ groups of variables.

##### **10.2 Kullback-Leibler (KL) divergence**

The Kullback-Leibler divergence is defined as.

<span style="color: blue;">

$$
\text{KL}[q||p] = \int q(z) \ln \frac{q(z)}{p(z)} dz
$$

</span>

Properties are:

1. Identify of inderscernibles:
$$
\text{KL}[q|p] = 0 \Rightarrow p=q
$$

2. Non-negativity:
$$
\text{KL}[q||p] \geq 0
$$

3. Asymmetry:
$$
\text{KL}[q||p] \neq \text{KL}[p||q]
$$

4. Does not satisfy triangle inequality:
$$
\text{KL}[q||p] \nleq \text{KL}[q||r] + \text{KL}[r||p]
$$

> **Which direction matters:** $\text{KL}[q\|p]$ (used in VI) is **zero-forcing**

> $q$ tends to underestimate the spread of $p$ and pick one mode. 

> $\text{KL}[p\|q]$ is **mass-covering** where $q$ tries to cover all modes of $p$.

> VI uses $\text{KL}[q\|p]$ because it only requires evaluating $\ln p(D,z)$, not $p(z|D)$.

##### **10.3 Mean-field / factorised variational family**



**Mean-field assumption:** $q(\theta) = \prod_i q_i(\theta_i)$ - all parameters are independent under $q$.

The mean-field family $\mathcal{Q}_1$ is a strict subset of all Gaussians, as it only contains Gaussians with diagonal covariance. A general Gaussian posterior with correlations between parameters cannot be represented exactly in $\mathcal{Q}_1$.

**Gaussian mean-field family** $\mathcal{Q}_1$:
$$q(w) = \prod_i \mathcal{N}(w_i | m_i, v_i)$$

| Quantity | Formula |
|---|---|
| **Entropy** | $H[q] = \sum_i \frac{1}{2}\ln(2\pi e\, v_i)$ |
| **KL to $\mathcal{N}(0,1)$** | $\text{KL}[\mathcal{N}(m,v) \| \mathcal{N}(0,1)] = \frac{1}{2}(v + m^2 - 1 - \ln v)$ |
| **$\mathbb{E}_q[w_i]$** | $m_i$ |
| **$\mathbb{E}_q[w_i^2]$** | $m_i^2 + v_i$ |
| **$\mathbb{E}_q[w_i w_j]$ ($i\neq j$)** | $m_i m_j$ (independence!) |

**Full-rank family** $\mathcal{Q}_2$:
$$q(w) = \mathcal{N}(w | m, S) \quad \text{(full covariance matrix } S\text{)}$$

> **Key inequality:** $\mathcal{Q}_1 \subseteq \mathcal{Q}_2 \Rightarrow \text{KL}[q^*_2 \| p(\cdot|y)] \leq \text{KL}[q^*_1 \| p(\cdot|y)]$

> Switching from mean-field to full-rank **can only improve** the approximation.

##### **10.4 CAVI - Coordinate Ascent Variational Inference**



For a factorised family $q(w) = \prod_j^J q(w_j)$, the **optimal factor** for $w_j$ is:

<span style="color: blue;">


$$\boxed{\ln q^*(w_j) \;\propto\; \mathbb{E}_{q_{-j}}\!\left[\ln p(D, w)\right] + K}$$

</span>

Where $q_{-j}$ means expectation over all factors *except* $w_j$.

**Algorithm:**

1. For $k=1,\dots, K$ we do:
$$
\ln q^*(w_k) = \mathbb{E}_{\prod_{i\neq k}q(w_i)}[\ln p(D, w)] + K
$$

2. Compute ELBO $\mathcal{L}[q]$ to monitor convergence.

**Exam pattern:** In CAVI we always:

> (1) write $\ln q^*(w_j) = \mathbb{E}_{q_{-j}}[\ln p(D,w)] + K$. 

> (2) drop all terms not involving $w_j$.

> (3) match the remaining functional form to a known distribution using the table above.


**KEY TRICKS for identifying functional form:**

| Shape of $\ln q^*_j$ (as function of $\theta_j$) | Distribution |
|---|---|
| $-\frac{1}{2v}(\theta_j - m)^2$ | $\mathcal{N}(\theta_j \mid m, v)$ |
| $(a-1)\ln \theta_j - b\theta_j$ | $\text{Gamma}(\theta_j \mid a, b)$ |
| $\sum_k (a_k - 1)\ln \pi_k$ | $\text{Dir}(\pi \mid a)$ |

Because for CAVI examples, we usually get a joint distribution and then want to approximate the resulting posterior using factorized distribution $q(w)$, then we use general CAVI update rules, and the optimal solution for a factorized distribution is then found by taking the log of the optimal factor and then identifying the resulting functional form with a known distribution, which is what the table above is for.

CAVI example from slides showed that join distribution model:
$$
\ln p(y,w) = \log p(y|w) + \log p(w) = -w_1^2 - \frac{1}{2}w_2^2 + w_1w_2 +6w_1 -3w_2
$$
Where we wanted to apprxoimate the resulting posterior using $q(w)=q(w_1)q(w_2)$ as factorized distribution, we then used CAVI update rule to get:
$$
\ln q^*(w_1) = \mathbb{E}_{q(w_2)}[\ln p(y,w)] + K = -w_1^2 + w_1 \mathbb{E}_{q(w_2)}[w_2] + 6w_1 + K
$$
$$
\ln q^*(w_2) = \mathbb{E}_{q(w_1)}[\ln p(y,w)] + K = -\frac{1}{2}w_2^2 + w_2 \mathbb{E}_{q(w_1)}[w_1] - 3w_2 + K
$$
Matching the functional forms we would arrive at:
$$
q(w_1) = \mathcal{N}(w_1 | 3 + \frac{1}{2}m_2, \frac{1}{2})
$$
$$
q(w_2) = \mathcal{N}(w_2 | m_1 - 3, 1)
$$

##### **10.5 Expected log-likelihood - exam computation pattern**



For Gaussian likelihood $p(y|w) = \mathcal{N}(y \mid w^T\Phi^T, \sigma^2 I)$ with $q(w) = \mathcal{N}(w|m,S)$:

$$\mathbb{E}_q[\ln p(y|w)] = -\frac{1}{2}\ln(2\pi\sigma^2) - \frac{1}{2\sigma^2}\sum_n \mathbb{E}_q[(y_n - w^T\phi_n)^2]$$

Where:
$$\mathbb{E}_q[(y_n - w^T\phi_n)^2] = (y_n - m^T\phi_n)^2 + \phi_n^T S \phi_n$$

**For bilinear model** $f = w_1 w_2$ with mean-field $q = \mathcal{N}(w_1|m_1,v_1)\mathcal{N}(w_2|m_2,v_2)$:
$$\mathbb{E}_q[f] = m_1 m_2, \qquad \mathbb{E}_q[f^2] = (m_1^2+v_1)(m_2^2+v_2)$$

$$\mathbb{E}_q[(y-f)^2] = y^2 - 2y\,m_1 m_2 + (m_1^2+v_1)(m_2^2+v_2)$$

<span style="color: red;">

Where $\phi_n = \phi(x_n)$ is the feature vector for the $n$-th data point, 
$m$ is the variational mean, $S$ is the variational covariance, and the second 
term $\phi_n^T S \phi_n$ captures the uncertainty in $w$ under $q$.

</span>

##### **10.5.1 Expected log-likelihood - the universal recipe**



> **Core insight:** No matter what $f(w)$ looks like, the computation always reduces to finding $\mathbb{E}_q[f]$ and $\mathbb{E}_q[f^2]$, then applying two moment identities for Gaussians.


**Step 0 - Always start here**

For **any** Gaussian likelihood $p(y \mid w) = \mathcal{N}(y \mid f(w), \sigma^2)$:

$$\mathbb{E}_q[\ln p(y \mid w)] = -\frac{1}{2}\ln(2\pi\sigma^2) - \frac{1}{2\sigma^2}\,\mathbb{E}_q\!\left[(y - f(w))^2\right]$$

Expand the square using linearity of expectation:

$$\mathbb{E}_q[(y - f(w))^2] = y^2 - 2y\,\mathbb{E}_q[f] + \mathbb{E}_q[f^2]$$

**You only ever need two numbers: $\mathbb{E}_q[f]$ and $\mathbb{E}_q[f^2]$.**

**Step 1 - The two moment identities (memorise these)**

For $q(w_i) = \mathcal{N}(w_i \mid m_i, v_i)$:

$$\boxed{\mathbb{E}_q[w_i] = m_i}$$
$$\boxed{\mathbb{E}_q[w_i^2] = m_i^2 + v_i}$$

For **mean-field** $q = \prod_i \mathcal{N}(w_i \mid m_i, v_i)$, independence gives:

$$\boxed{\mathbb{E}_q[w_i w_j] = m_i m_j \quad \text{for } i \neq j}$$
$$\boxed{\mathbb{E}_q[w_i^2 w_j^2] = (m_i^2 + v_i)(m_j^2 + v_j) \quad \text{for } i \neq j}$$

<span style="color: red;">

The independence factorisation only holds for mean-field. For a full-rank Gaussian $q(w) = \mathcal{N}(w \mid m, \Sigma)$, cross terms become $\mathbb{E}[w_i w_j] = m_i m_j + \Sigma_{ij}$ - the off-diagonal covariance contributes.

</span>

**Step 2 - Apply to any $f(w)$: worked examples**

**Case A - Linear model:** $f = w^T\phi$ with full-rank $q(w) = \mathcal{N}(m, S)$

$$\mathbb{E}_q[f] = m^T\phi$$
$$\mathbb{E}_q[f^2] = (m^T\phi)^2 + \phi^T S \phi$$

So:
$$\mathbb{E}_q[(y - f)^2] = (y - m^T\phi)^2 + \phi^T S \phi$$

The $\phi^T S \phi$ term is the **uncertainty penalty** - it vanishes only if $S = 0$ (no uncertainty).

**Case B - Bilinear model (2024 exam):** $f = w_1 w_2$ with mean-field

$$\mathbb{E}_q[f] = m_1 m_2$$
$$\mathbb{E}_q[f^2] = (m_1^2 + v_1)(m_2^2 + v_2)$$

So:
$$\mathbb{E}_q[(y - f)^2] = y^2 - 2y\,m_1 m_2 + (m_1^2 + v_1)(m_2^2 + v_2)$$


**Case C - Affine model:** $f = w_1 + w_2 x$ with mean-field

$$\mathbb{E}_q[f] = m_1 + m_2 x$$
$$\mathbb{E}_q[f^2] = (m_1 + m_2 x)^2 + v_1 + v_2 x^2$$

The cross term $2m_1 m_2 x$ comes from expanding the square; the uncertainty terms $v_1 + v_2 x^2$ come from $\mathbb{E}[w_i^2] - m_i^2 = v_i$, scaled by the coefficient of $w_i$ squared.

So:
$$\mathbb{E}_q[(y - f)^2] = (y - m_1 - m_2 x)^2 + v_1 + v_2 x^2$$


**Case D - Linear + bilinear (possible exam twist):** $f = w_1 + w_1 w_2$ with mean-field

$$\mathbb{E}_q[f] = m_1 + m_1 m_2 = m_1(1 + m_2)$$
$$\mathbb{E}_q[f^2] = \mathbb{E}_q[w_1^2 + 2w_1^2 w_2 + w_1^2 w_2^2]$$
$$= (m_1^2 + v_1) + 2(m_1^2 + v_1)m_2 + (m_1^2 + v_1)(m_2^2 + v_2)$$
$$= (m_1^2 + v_1)(1 + 2m_2 + m_2^2 + v_2)$$

This shows the general method: **expand $f^2$ as a polynomial in $w_i$, then apply the identities term by term**.


**Step 3 - Sanity checks**

Before writing down your final answer, verify:

1. **Dimension check:** Does each term have the right units? (both sides should be scalars)
2. **Zero variance limit:** If $v_i \to 0$, does the result collapse to $(y - f(m))^2$? It should.
3. **Symmetry:** If $w_1$ and $w_2$ play symmetric roles in the model, do $m_1, v_1$ and $m_2, v_2$ appear symmetrically in the result?


Quick-reference table

| $f(w)$ | $\mathbb{E}_q[f]$ | $\mathbb{E}_q[f^2]$ | Variational family |
|--------|------------------|---------------------|--------------------|
| $w^T\phi$ | $m^T\phi$ | $(m^T\phi)^2 + \phi^T S\phi$ | Full-rank $\mathcal{N}(m, S)$ |
| $w^T\phi$ | $m^T\phi$ | $(m^T\phi)^2 + \sum_i v_i \phi_i^2$ | Mean-field |
| $w_1 w_2$ | $m_1 m_2$ | $(m_1^2 + v_1)(m_2^2 + v_2)$ | Mean-field |
| $w_1 + w_2 x$ | $m_1 + m_2 x$ | $(m_1 + m_2 x)^2 + v_1 + v_2 x^2$ | Mean-field |

##### **10.6 Gaussian Mixture Model (GMM)**



**Marginalized form** ($K$ components):

<span style="color: blue;">

$$p(x_n \mid \pi, m, \Lambda) = \sum_{k=1}^K \pi_k\, \mathcal{N}(x_n \mid m_k, \Lambda_k^{-1})$$

</span>

<span style="color: red;">

Where $\pi_k$ are the **mixing weights** ($\sum_k \pi_k = 1$, $\pi_k \geq 0$), 
$m_k$ are the **component means**, $\Lambda_k$ are the **precision matrices** 
(inverse covariance), and $K$ is the number of mixture components.

</span>

**Latent variable form:**
$$
p(x_n|z_n) = \prod_{k=1}^K \mathcal{N}(x|\mu_k, \Lambda_k ^{-1})^{z_{nk}}
$$
$$
p(z_n) = \text{Cat}(z|\pi) = \prod_{k=1}^K \pi_k^{z_{nk}}
$$

Latent variables $z_n$ are variables that we cannot observe directly.

Both forms are connected by:
$$p(x_n) = \sum_k p(x_n|z_n=k)\,p(z_n=k)$$

**Bayesian GMM** adds conjugate priors:
$$\pi \sim \text{Dir}(\alpha_0 \mathbf{1}), \qquad m_k|\Lambda_k \sim \mathcal{N}(m_0, [\beta_0 \Lambda_k]^{-1}), \qquad \Lambda_k \sim \mathcal{W}(W_0, \nu_0)$$

**Variational family for Bayesian GMM:**
$$q(Z, \pi, m, \Lambda) = q(Z)\,q(\pi, m, \Lambda)$$
Only one factorisation assumption needed - the latent cluster assignments $Z$ are separated from the parameters.

**Why posterior is intractable:** summing over all $K^N$ configurations is exponential in $N$.

We have different models:

<span style="color: blue;">

$$
\pi \sim \text{Dirichlet}(\alpha_0)
$$
$$
\Lambda_k \sim \text{Wishart}(W_0, \nu_0)
$$
$$
\mu_k | \Lambda_k \sim \text{Normal}(m_0, [\beta_0 \Lambda_k]^{-1})
$$
$$
z_n | \pi \sim \text{Categorical}(\pi)
$$
$$
x_n | \mu, \Lambda, z_n \sim \text{Normal}(\mu_{z_n}, \Lambda_{z_n}^{-1})
$$

</span>

##### **10.6.5 Fitting GMM using maximum likelihood**

1. Initialize all parameters $\pi_k$, $\mu_k$, $\Sigma_k$ for $k=1,\dots, K$.

2. Repeat until convergence:

> Expectation-setp:

<span style="color: blue;">

$$
\gamma_{nk} = \frac{\pi_k \mathcal{N}(x_n | \mu_k, \Sigma_k)}{\sum_{j=1}^K \pi_j \mathcal{N}(x_n | \mu_j, \Sigma_j)}
$$

</span>

<span style="color: red;">

Where $\gamma_{nk}$ is the **responsibility** - the posterior probability that 
component $k$ generated data point $x_n$. In the M-step, $N_k = \sum_n \gamma_{nk}$ 
is the **effective number of points** assigned to component $k$.

</span>

> Maximization-step:

<span style="color: blue;">

$$
N_k = \sum_{n=1}^N \gamma_{nk}, \qquad \pi_k^* = \frac{N_k}{N}, \qquad \mu_k^* = \frac{1}{N_k} \sum_{n=1}^N \gamma_{nk} x_n, \qquad \Sigma_k^* = \frac{1}{N_k} \sum_{n=1}^N \gamma_{nk} (x_n - \mu_k^*)(x_n - \mu_k^*)^T
$$

</span>

##### **10.7 Key distributions in VI**



**Dirichlet distribution** $\text{Dir}(\pi \mid \alpha)$:

> Domain: $K$-dimensional probability simplex ($\pi_k \geq 0$, $\sum_k \pi_k = 1$)

> $\text{Dir}(\pi| \alpha) = \frac{\Gamma(\sum_k \alpha_k)}{\prod_k \Gamma(\alpha_k)} \prod_k \pi_k^{\alpha_k - 1}$ where $\alpha = \sum_{k=1}^K \alpha_k$

> Mean: $\mathbb{E}[\pi_k] = \alpha_k / \sum_j \alpha_j$

> Called "distribution over distributions" so a sample is a valid probability vector

> Conjugate prior for categorical/multinomial likelihoods

**Gamma distribution** $\text{Gamma}(\tau \mid a, b)$:
> Domain: $\tau > 0$

> $p(\tau) \propto \tau^{a-1} e^{-b\tau}$, and $\mathbb{E}[\tau] = a/b$

> Conjugate prior for precision (inverse variance) of Gaussian

> $\ln p(\tau) = (a-1)\ln\tau - b\tau + \text{const}$

**Wishart distribution** $\mathcal{W}(\Lambda \mid W, \nu)$:
> Domain: positive definite matrices

> $\mathcal{W}(\Lambda | W_0, \nu_0) = B | \Lambda |^{(\nu - D - 1)/2} \exp\left(-\frac{1}{2} \text{tr}(W^{-1} \Lambda)\right)$ where $B$ is a normalising constant

> Mean: $\mathbb{E}[\Lambda] = \nu W$ with $\mathbb{E}[\Lambda^{-1}] = W^{-1} / (\nu - D - 1)$ for $\nu > D + 1$

> Conjugate prior for precision matrices of multivariate Gaussians

##### **10.8 Variational inference for mixture models**

> *(This is CAVI applied to the Bayesian GMM from 10.6 - the same update rule $\ln q^* \propto \mathbb{E}_{q_{-j}}[\ln p]$ but now applied to mixture model variables.)*

The goal is to compute parameters for the mixture model:
$$
p(Z, \mu, \Lambda, \pi | X) = \frac{p(X|Z, \mu, \Lambda) p(Z|\pi) p(\pi) p(\mu|\Lambda) p(\Lambda)}{p(X)}
$$
Use VI with factorized approximation:
$$
q(Z, \mu, \Lambda, \pi) = q(Z) q(\pi) q(\mu, \Lambda)
$$
Iterative algorithm to minimze the KL divergence:


$$\ln q(\boldsymbol{Z}) \propto \mathbb{E}_{q(\boldsymbol{\mu}, \boldsymbol{\Lambda}, \boldsymbol{\pi})}[\ln p(\boldsymbol{X}, \boldsymbol{Z}, \boldsymbol{\mu}, \boldsymbol{\Lambda}, \boldsymbol{\pi})]$$

$$\ln q(\boldsymbol{\pi}, \boldsymbol{\mu}, \boldsymbol{\Lambda}) \propto \mathbb{E}_{q(\boldsymbol{Z})}[\ln p(\boldsymbol{X}, \boldsymbol{Z}, \boldsymbol{\mu}, \boldsymbol{\Lambda}, \boldsymbol{\pi})]$$

The resulting approximation is:

$$q(\boldsymbol{Z}, \boldsymbol{\pi}, \boldsymbol{\mu}, \boldsymbol{\Lambda}) = q(\boldsymbol{Z})\, q(\boldsymbol{\pi}, \boldsymbol{\mu}, \boldsymbol{\Lambda})$$

$$= \underbrace{\prod_{n=1}^{N} \text{Categorical}(z_n | r_n)}_{q(\boldsymbol{Z})} \quad \underbrace{\text{Dir}(\boldsymbol{\pi} | \boldsymbol{\alpha})}_{q(\boldsymbol{\pi})} \quad \underbrace{\prod_{k=1}^{K} \mathcal{N}\!\left(\boldsymbol{\mu}_k \,\big|\, \boldsymbol{m}_k,\, \left[\beta_k \boldsymbol{\Lambda}_k^{-1}\right]\right) \mathcal{W}(\boldsymbol{\Lambda}_k | W_k, \nu_k)}_{q(\boldsymbol{\mu}, \boldsymbol{\Lambda})}$$

<a id='week11'></a>

<div class="alert alert-block alert-info">

### **Week 11 | Black-box variational inference (BBVI)**

Free-form vs fixed-form VI, reparameterization trick, minibatching, VAEs, scaling GPs with inducing points

</div>

##### **11.1 Recap: the VI problem**


From week 10, the goal of variational inference is:

<span style="color: blue;">

$$q^* = \arg\min_{q \in \mathcal{Q}} \text{KL}[q(\theta) \| p(\theta|y)]$$

</span>

Which is equivalent to maximising the **ELBO**:

<span style="color: blue;">


$$\mathcal{L}[q] = \mathbb{E}_q[\log p(y, \theta)] - \mathbb{E}_q[\log q(\theta)] = \underbrace{\mathbb{E}_q[\log p(y|\theta)]}_{\text{expected log-lik}} - \underbrace{\text{KL}[q(\theta) \| p(\theta)]}_{\text{KL to prior}}$$

</span>

**Classic VI** (for instance CAVI from week 10) computes $\mathbb{E}_q[\log p(y, \theta)]$ **analytically** using conjugacy. This is fast but restrictive but we must re-derive everything for every new model.

**Black-box VI** (BBVI) estimates the expectation with Monte Carlo sampling, so it works for *any* model where you can evaluate $\log p(y, \theta)$.


##### **11.1.5 Free-form VI**

> *(This is the bridge between CAVI and BBVI, instead of updating $q$ factors analytically, we parametrize $q$ and optimise the ELBO directly with gradients.)*

If we consider a model $p(y,w)$ where $w\in \mathbb{R}^D$ and we choose a full-rank Gaussian family $q(w)=\mathcal{N}(w|m,V)$ such that the variational family $\mathcal{Q}$ consists of all multivariate Gaussian distributions, then $q_\psi$ is now *parametrized* by the *variational parameters* $\psi = \{\boldsymbol{m}, \boldsymbol{V}\}$

$$\mathcal{L}[q_\psi] = \mathbb{E}_{q_\psi}[\ln p(\boldsymbol{y}, \boldsymbol{w})] - \mathbb{E}_{q_\psi}[\ln q_\psi(\boldsymbol{w})]$$

$$= \mathbb{E}_{\mathcal{N}(\boldsymbol{w}|\boldsymbol{m}, \boldsymbol{V})}[\ln p(\boldsymbol{y}, \boldsymbol{w})] - \mathbb{E}_{\mathcal{N}(\boldsymbol{w}|\boldsymbol{m}, \boldsymbol{V})}[\ln \mathcal{N}(\boldsymbol{w}|\boldsymbol{m}, \boldsymbol{V})]$$

Fitting the approximation using gradient-based methods:

$$q^* = \arg\max_{q \in \mathcal{Q}} \mathcal{L}[q] \iff \psi^* = \arg\max_{\psi}\, \mathcal{L}[q_\psi] \iff \boldsymbol{m}^*, \boldsymbol{V}^* = \arg\max_{\boldsymbol{m}, \boldsymbol{V}}\, \mathcal{L}[q_\psi]$$

##### **11.2 Why we need the reparameterization trick**


The ELBO gradient with respect to variational parameters $\lambda$ is:

<span style="color: blue;">

$$\nabla_\lambda \mathcal{L}[q_\lambda] = \nabla_\lambda \mathbb{E}_{q_\lambda(\theta)}[\log p(y, \theta)] - \nabla_\lambda \mathbb{E}_{q_\lambda(\theta)}[\log q_\lambda(\theta)]$$

</span>

The entropy term on the right is often analytically tractable. The problem is the **first term** the expectation is taken with respect to a distribution that itself depends on $\lambda$, so we cannot simply push the gradient inside the expectation:

$$\nabla_\lambda \mathbb{E}_{q_\lambda(\theta)}[f(\theta)] \neq \mathbb{E}_{q_\lambda(\theta)}[\nabla_\lambda f(\theta)]$$

(because the distribution of $\theta$ also changes with $\lambda$).


**The reparameterization trick** solves this by changing variables: instead of sampling $\theta \sim q_\lambda(\theta)$ directly, write

$$\theta = g(\epsilon, \lambda), \quad \epsilon \sim p(\epsilon) = \mathcal{N}(0, I)$$

For a mean-field Gaussian $q(w) = \prod_i \mathcal{N}(w_i | m_i, v_i)$:

$$w_i = m_i + \sqrt{v_i}\,\epsilon_i, \quad \epsilon_i \sim \mathcal{N}(0, 1)$$

Now the expectation is over a **fixed** distribution $p(\epsilon)$:

$$\mathbb{E}_{q_\lambda(w)}[\log p(y, w)] = \mathbb{E}_{p(\epsilon)}[\log p(y, g(\epsilon, \lambda))]$$

Andt then we can push the gradient inside:

<span style="color: blue;">


$$\nabla_\lambda \mathbb{E}_{q_\lambda(w)}[\log p(y,w)] \approx \frac{1}{S}\sum_{s=1}^S \nabla_\lambda \log p(y, w_{\epsilon_s}), \quad w_{\epsilon_s} = m + \sqrt{v} \circ \epsilon_s$$

</span>

<span style="color: red;">

Where $m_i$ is the variational mean for dimension $i$, $v_i$ is the variational 
variance, $\epsilon_i \sim \mathcal{N}(0,1)$ is the fixed noise variable, and 
$\circ$ denotes element-wise multiplication. The key point is that $\lambda = \{m, v\}$  are now the only things the gradient flows through.

</span>

> **Key insight:** The reparameterization trick moves the randomness into a fixed noise variable $\epsilon$, making gradient-based optimisation of the variational parameters possible via automatic differentiation.

> REMEMBER that $\circ$ means element-wise multiplication.

##### **11.3 The BBVI algorithm**


**Setup:** Mean-field Gaussian variational family with variational parameters $\lambda = \{m, v\}$ (means and variances for each dimension).

**Entropy term** (always analytical for Gaussians):

<span style="color: blue;">

$$H[q] = \frac{1}{2}\sum_{i=1}^D \ln(2\pi e\, v_i) = \frac{D}{2}\ln(2\pi e) + \frac{1}{2}\sum_{i=1}^D \ln v_i$$

</span>

Where $\pi$ is `jnp.pi` and $e$ is `jnp.e` in JAX.

**ELBO estimate (the BBVI objective):**

$$\mathcal{L}[q] \approx \underbrace{\frac{1}{S}\sum_{s=1}^S \log p(y, w^{(s)})}_{\text{MC estimate of } \mathbb{E}_q[\log p(y,w)]} + H[q]$$

Where $w^{(s)} = m + \sqrt{v} \circ \epsilon^{(s)}$, $\epsilon^{(s)} \sim \mathcal{N}(0, I)$.

**Algorithm:**

1. Initialise variational parameters $\lambda = \{m, v\}$ (tip: store log v to keep variances positive during optimisation).

2. Repeat until ELBO converges:

   a. Sample $\epsilon^{(1)}, ..., \epsilon^{(S)} \sim \mathcal{N}(0, I)$

   b. Reparameterise: $w^{(s)} = m + \sqrt{v} \circ \epsilon^{(s)}$

   c. Estimate ELBO:
      $\mathcal{L} \approx \frac{1}{S} \sum_{s=1}^S \log p(y, w^{(s)}) + H[q]$

   d. Compute gradient $\nabla_\lambda \mathcal{L}$ using autodiff (JAX/PyTorch)

   e. Update $\lambda$ using Adam (or another gradient-based optimizer)

3. Monitor ELBO for convergence


> **Parameter encoding trick (exam!):** Optimise in unconstrained space. Variances $v_i > 0$ are stored as $\log v_i \in \mathbb{R}$, so the optimizer can take unconstrained gradient steps. When needed, recover $v_i = \exp(\log v_i)$.


##### **11.4 Minibatching for large datasets**



For $N$ i.i.d. data points, the log-joint decomposes:

$$\log p(y, w) = \sum_{n=1}^N \log p(y_n | w) + \log p(w)$$

Evaluating all $N$ terms each iteration is expensive. **Minibatching** uses a random subset $\mathcal{M}$ of size $M \ll N$:

$$\mathbb{E}_q[\log p(y, w)] \approx \frac{N}{M} \sum_{n \in \mathcal{M}} \mathbb{E}_q[\log p(y_n|w)] + \mathbb{E}_q[\log p(w)]$$

The factor $N/M$ corrects for the smaller batch, this gives an **unbiased** estimator of the full expected log-joint when $\mathcal{M}$ is resampled each iteration.

| Setting | Trade-off |
|---------|-----------|
| Small $M$ | Noisy gradient, but fast per iteration |
| Large $M$ | More stable gradient, but slower per iteration |
| Small $S$ (MC samples) | Fast but noisy ELBO estimate |
| Large $S$ | Slow but stable, usually 10–20 is enough |


##### **11.5 Mean-field approximation with what it gets wrong**

For a target distribution with covariance $V$ and a mean-field approximation with diagonal $\hat{V}$, the optimal variational variances satisfy:

<span style="color: blue;">


$$\hat{V}_{ii} = \frac{1}{(V^{-1})_{ii}}$$

</span>

Note that this is the inverse of the *diagonal of the precision matrix* so **not** the diagonal of the covariance matrix $V_{ii}$.

For a 2D Gaussian with correlation $\rho$: $V = \begin{bmatrix}1 & \rho \\ \rho & 1\end{bmatrix}$:

> $V^{-1} = \frac{1}{1-\rho^2}\begin{bmatrix}1 & -\rho \\ -\rho & 1\end{bmatrix}$, so $(V^{-1})_{11} = \frac{1}{1-\rho^2}$

> $\hat{V}_{11} = 1 - \rho^2$

The mean-field variance **underestimates** the true marginal variance $V_{11} = 1$ whenever $\rho \neq 0$. The stronger the correlation, the worse the underestimation.

> **Key takeaway:** Mean-field variational families cannot capture correlations between parameters by construction. They systematically underestimate marginal variances when the true posterior has off-diagonal covariance structure. This is a known limitation of CAVI and BBVI with mean-field families.


##### **11.6 Free-form vs fixed-form variational inference**



| Type | Description | Example |
|------|-------------|---------|
| **Fixed-form (parametric)** | $q$ restricted to a parametric family | Mean-field Gaussian, full-rank Gaussian |
| **Free-form (non-parametric)** | $q$ can be any distribution; optimise functional directly | CAVI (updates $q_j$ directly), normalizing flows |

**CAVI** (from week 10) is technically free-form within the factorised family where the optimal $q_j^*$ has no pre-specified distribution family, so its form is determined by the model.

**BBVI** is fixed-form where we fix $q$ to be Gaussian and optimise the parameters $\{m, v\}$.


##### **11.7 Variational autoencoders (VAEs)**



A VAE is BBVI applied to a **latent variable model** where:

> **Model (decoder):** $x_n | z_n \sim \mathcal{N}(f_\theta(z_n), \sigma^2 I)$, $\quad z_n \sim \mathcal{N}(0, I)$

> **Variational approximation (encoder):** $q(z_n) = \prod_k \mathcal{N}(z_{nk} | g^1_\phi(x_n), \exp(g^2_\phi(x_n)))$

> The slides denote $m(x_n) = g^1_\phi(x_n)$ and $\ln v(x_n) = g^2_\phi(x_n)$ with $q(z_n) = \prod_k \mathcal{N}(z_{nk} | m_k(x_n), v_k(x_n))$.

Where $f_\theta$ (decoder) and $g_\phi$ (encoder) are neural networks.

**Objective** is to maximise the ELBO with respect to both $\theta$ and $\phi$ in order to optimize the ELBO so we can fit the model with NN parameters:

<span style="color: blue;">


$$\arg\max_{\theta, \phi} \mathcal{L}[q] = \mathbb{E}_{q_\phi(z|x)}[\log p_\theta(x|z)] - \text{KL}[q_\phi(z|x) \| p(z)]$$

</span>

The reparameterization trick enables gradients with respect to $\phi$ (the encoder parameters).

> **VAE = non-linear factor model + variational inference.** The encoder network outputs the variational parameters (mean and log-variance of $q(z|x)$), the decoder network defines the likelihood. This is why VAEs are generative: at test time, sample $z \sim \mathcal{N}(0,I)$ and decode.


##### **11.8 Scaling GPs with inducing points**




Standard GP inference is $\mathcal{O}(N^3)$ - intractable for large $N$. **Sparse GP** methods introduce $M \ll N$ **inducing points** $\{z_i\}$ with function values $u_i = f(z_i)$.

**Variational approximation:**

$$q(f) = \int p(f|u)\, q(u)\, du, \quad q(u) = \mathcal{N}(u | m_u, S_u)$$

**Collapsed lower bound** (optimise over $q(u)$ analytically):

$$\mathcal{L}[q] = \log \mathcal{N}(y | 0,\; K_{fu}K_{uu}^{-1}K_{uf} + \sigma^2 I) - \frac{1}{2\sigma^2}\text{trace}(K_{ff} - K_{fu}K_{uu}^{-1}K_{uf})$$

Where $K_{fu}$ is the $N \times M$ cross-covariance matrix between training and inducing points.

**Posterior over inducing points** (after optimizing bound):

<span style="color: blue;">

$$S_u^{-1} = \frac{1}{\sigma^2}K_{uu}^{-1}K_{uf}K_{fu}K_{uu}^{-1} + K_{uu}^{-1}, \quad m_u = \frac{1}{\sigma^2}S_u K_{uu}^{-1}K_{uf}y$$

</span>

**Approximate posterior for any point** $f$:

<span style="color: blue;">

$$m_f = K_{fu}K_{uu}^{-1}m_u, \quad S_f = K_{ff} + K_{fu}K_{uu}^{-1}(S_u - K_{uu})K_{uu}^{-1}K_{uf}$$

</span>

<span style="color: red;">

Where $K_{uu} \in \mathbb{R}^{M \times M}$ is the kernel matrix between inducing 
points, $K_{fu} \in \mathbb{R}^{N \times M}$ is the cross-covariance between 
training points and inducing points, $m_u$ and $S_u$ are the variational mean 
and covariance of the inducing point values $u$, and $M \ll N$ is the number 
of inducing points.

</span>

**Cost:** $\mathcal{O}(NM^2)$ instead of $\mathcal{O}(N^3)$ which is huge win for large datasets.

**The trace term** $\frac{1}{2\sigma^2}\text{trace}(K_{ff} - K_{fu}K_{uu}^{-1}K_{uf})$ is a regularizer that penalises placing inducing points away from the data - it's the "cost" of approximation.


##### **11.9 BBVI vs CAVI comparison**



<span style="color:  blue;">

| Aspect | CAVI (Week 10) | BBVI (Week 11) |
|--------|---------------|----------------|
| ELBO computation | Analytical | Monte Carlo (noisy) |
| Applicable to | Conjugate models only | Any model with differentiable log-joint |
| Gradient | Closed-form updates | Stochastic gradient via reparameterization |
| Speed | Fast per iteration | Slower per iteration, but scales to large data via minibatching |
| Complexity | Must re-derive updates per model | Plug-and-play: just implement log-lik + log-prior |

</span>

<a id='week12'></a>

<div class="alert alert-block alert-info">


### **Week 12 | Bayesian neural networks**

MAP inference, deep ensembles, last-layer Laplace approximation (LLLA), Monte Carlo dropout, mean-field BNNs, heteroscedastic models

</div>

##### **12.1 Why Bayesian methods for neural networks?**


> *(The posterior predictive integral is the same as always - the challenge unique to NNs is that $p(w|y)$ is extremely high-dimensional, multimodal, and non-Gaussian,  making every approximation method from weeks 3–11 harder to apply.)*


Standard (deterministic) neural networks output point predictions with no uncertainty estimates. A Bayesian neural network (BNN) treats the weights $w$ as random variables and computes the **posterior predictive distribution**:

$$p(y^*|y, x^*) = \int p(y^*|w, x^*)\, p(w|y)\, dw$$

**Challenges unique to neural networks:**

> Millions of parameters → posterior geometry is enormously complex

> Non-linear activations → posterior is highly non-Gaussian, multi-modal

> Scale of data → MCMC is impractical

> Standard matrix inversion for full Laplace approx is $\mathcal{O}(D^3)$ so prohibitive for $D > 10^4$



##### **12.2 Spectrum of approximations**


From cheapest to most expensive / most accurate:

```
MAP (plug-in)  →  LLLA  →  MC Dropout  →  Mean-field VI  →  Full Laplace  →  MCMC
fast/cheap                                                          slow/accurate
```

All methods share the same prediction formula at test time:

$$p(y^*|y,x^*) \approx \frac{1}{S}\sum_{i=1}^S p(y^*|w^{(i)}, x^*)$$

They differ only in **how the samples** $w^{(i)}$ are obtained.


##### **12.3 MAP inference (the baseline)**


Maximise the log-joint with respect to $w$:

$$\hat{w}_{\text{MAP}} = \arg\max_w \log p(y|w) + \log p(w)$$

With Gaussian prior $p(w) = \mathcal{N}(0, \alpha^{-1}I)$, this is identical to training with **L2 regularisation** (weight decay):

$$\hat{w}_{\text{MAP}} = \arg\min_w \mathcal{L}_{\text{CE/MSE}}(w) + \alpha \|w\|^2$$

> **Key insight:** Every time we train a neural network with weight decay, we are implicitly computing a MAP estimate under a Gaussian prior. The regularisation  'strength $\alpha$ is the prior precision.


MAP corresponds to a **degenerate posterior approximation**:

$$q_{\text{MAP}}(w) = \delta(w - \hat{w}_{\text{MAP}})$$

So the predictive distribution is just the plug-in: $p(y^*|y,x^*) \approx p(y^*|\hat{w}_{\text{MAP}}, x^*)$. This gives **no epistemic uncertainty** and is typically **overconfident** out-of-distribution.


##### **12.4 Deep ensembles**


Train $S$ independent networks from different random initialisations:

<span style="color: blue;">

$$q_{\text{DE}}(w) = \frac{1}{S}\sum_{i=1}^S \delta(w - w^{(i)})$$

</span>

**Posterior predictive:**

<span style="color: blue;">

$$p(y^*|y,x^*) \approx \frac{1}{S}\sum_{i=1}^S p(y^*|w^{(i)}, x^*)$$

</span>

For Gaussian regression with heteroscedastic outputs $[\mu_{w^{(i)}}(x^*), \sigma^2_{w^{(i)}}(x^*)]$:

$$p(y^*|y,x^*) \approx \frac{1}{S}\sum_{i=1}^S \mathcal{N}(y^* | \mu_{w^{(i)}}(x^*),\; e^{f_2^{(i)}(x^*)})$$

This is a **Gaussian mixture model** with $S$ components, equal weights $1/S$, and the $i$-th network's outputs as means and (log-)variances.

> **Intuition:** Different initialisations can converge to different local minima of the loss, particularly for under-determined problems. These pseudo-samples provide uncertainty estimates by diversity of predictions - not from any explicit probabilistic model of the posterior.

**Pros:** Easy to implement, strong empirical performance, parallelisable.  
**Cons:** $S \times$ training cost, no principled posterior interpretation.

The mean and variance of this mixture are:

$$\mu^* = \frac{1}{S}\sum_{i=1}^S \mu_{w^{(i)}}(x^*)$$

$$(\sigma^*)^2 = \frac{1}{S}\sum_{i=1}^S \left[e^{f_2^{(i)}(x^*)} + \mu_{w^{(i)}}(x^*)^2\right] - (\mu^*)^2$$

<span style="color: red;">

Where $f_1^{(i)}(x^*)$ is the mean output of the $i$-th network, 
$e^{f_2^{(i)}(x^*)}$ is its predicted variance (log-parameterised to ensure 
positivity), and $S$ is the ensemble size.

</span>

> The variance decomposes into **aleatoric** (average predicted variances $e^{f_2^{(i)}}$) 
> and **epistemic** (variance of the predicted means across ensemble members) uncertainty.

##### **12.5 Last-layer Laplace approximation (LLLA)**



Computing the full Hessian of a deep network is $\mathcal{O}(D^3)$ is too expensive. The **LLLA** is a practical compromise:

1. Train the full network to convergence → obtain $\hat{w}_{\text{MAP}}$

2. Treat all layers **except the last** as fixed/deterministic

3. Apply the Laplace approximation **only to the last layer weights** $w_L = \{W_2, b_2\}$

**Why this works:** The last layer is a linear model on top of a learned feature representation $z_2 = f_{1:L-1}(x)$. This is exactly the Bayesian linear regression setup from week 4!

**Steps:**

1. Compute the MAP network $\hat{w}_{\text{MAP}}$ via gradient descent
2. Extract last-layer features: $z_2^{(n)} = f_{1:L-1}(x_n | \hat{w}_{1:L-1})$
3. Compute Hessian of log-joint w.r.t. last-layer weights only:

$$H_L = -\nabla^2_{w_L} \log p(y, w)|_{\hat{w}_{\text{MAP}}}$$

4. Laplace approximation for last-layer: $q(w_L) = \mathcal{N}(w_L | \hat{w}_L, H_L^{-1})$

5. Sample $w_L^{(s)} \sim q(w_L)$ and average predictions

**Key formula** (posterior covariance of last-layer for Gaussian regression):

<span style="color: blue;">

$$S_L = \left(\alpha I + \frac{1}{\sigma^2} Z^T Z\right)^{-1}$$

</span>

Where $Z$ is the matrix of last-layer features, identical to the Bayesian linear regression posterior!

> This is exactly $S = (\alpha I + \beta \Phi^T\Phi)^{-1}$ from section 3.4, with $Z$ playing the role of $\Phi$ (the design matrix of learned features) and $\beta = 1/\sigma^2$.

> **Post-hoc:** LLLA can be applied to **any already-trained network** without retraining. This is one of its biggest practical advantages.


##### **10.5 Expected log-likelihood - exam computation pattern**



For Gaussian likelihood $p(y|w) = \mathcal{N}(y \mid w^T\Phi^T, \sigma^2 I)$ with $q(w) = \mathcal{N}(w|m,S)$:

$$\mathbb{E}_q[\ln p(y|w)] = -\frac{1}{2}\ln(2\pi\sigma^2) - \frac{1}{2\sigma^2}\sum_n \mathbb{E}_q[(y_n - w^T\phi_n)^2]$$

Where:
$$\mathbb{E}_q[(y_n - w^T\phi_n)^2] = (y_n - m^T\phi_n)^2 + \phi_n^T S \phi_n$$

**For bilinear model** $f = w_1 w_2$ with mean-field $q = \mathcal{N}(w_1|m_1,v_1)\mathcal{N}(w_2|m_2,v_2)$:
$$\mathbb{E}_q[f] = m_1 m_2, \qquad \mathbb{E}_q[f^2] = (m_1^2+v_1)(m_2^2+v_2)$$

$$\mathbb{E}_q[(y-f)^2] = y^2 - 2y\,m_1 m_2 + (m_1^2+v_1)(m_2^2+v_2)$$

##### **12.6 Heteroscedastic neural network model**


> **Intuition:** When $f_2(x_n|w)$ is large (high predicted variance), the squared error term is downweighted but you pay a penalty of $\frac{1}{2}f_2$. The network  learns to predict high variance only when it genuinely helps reduce the total loss.


The network outputs **two values** per input: predicted mean and predicted log-variance of the noise.

<span style="color: blue;">


$$p(y_n|w, x_n) = \mathcal{N}(y_n | \underbrace{f_1(x_n|w)}_{\text{predicted mean}},\; \underbrace{e^{f_2(x_n|w)}}_{\text{predicted variance}})$$

</span>

The log-variance parameterisation guarantees $\sigma^2 > 0$ for any network output.

**Log-likelihood** (for optimisation):

<span style="color: blue;">

$$\log p(y_n|w,x_n) = -\frac{1}{2}f_2(x_n|w) - \frac{(y_n - f_1(x_n|w))^2}{2\,e^{f_2(x_n|w)}} + \text{const}$$

</span>

This is a loss that simultaneously fits the mean and learns the noise level - higher predicted variance automatically downweights the squared error.

**Homoscedastic vs. heteroscedastic:**

| Type | Model | Variance |
|------|-------|---------|
| **Homoscedastic** | $y_n\|w \sim \mathcal{N}(f_1(x_n\|w), \sigma^2)$ | Constant, $\sigma^2$ fixed |
| **Heteroscedastic** | $y_n\|w \sim \mathcal{N}(f_1(x_n\|w), e^{f_2(x_n\|w)})$ | Input-dependent |


##### **12.7 Log predictive density (LPD) evaluation metric**



To evaluate probabilistic predictions, use the **average log predictive density**:

<span style="color: blue;">

$$\text{LPD} = \frac{1}{N_{\text{test}}} \sum_{n=1}^{N_{\text{test}}} \log p(y_n^* | y, x_n^*)$$

</span>

For ensemble/sample-based predictions, estimate:

$$p(y_n^*|y, x_n^*) \approx \frac{1}{S}\sum_{s=1}^S p(y_n^*|w^{(s)}, x_n^*)$$

So:

$$\text{LPD} \approx \frac{1}{N_{\text{test}}} \sum_n \log \left[\frac{1}{S}\sum_s p(y_n^*|w^{(s)}, x_n^*)\right]$$

> **Note:** Take log outside the sum - this is the log of a *mixture* and cannot be simplified further unless the components are very separated. 


##### **12.8 Monte Carlo dropout**



During training, apply dropout (randomly zero out activations with probability $p$) as usual. During test time, **keep dropout active** and run $S$ forward passes.

**Interpretation:** MC dropout can be interpreted as approximate Bayesian inference with a specific implicit variational distribution (Gal & Ghahramani, 2016). Each forward pass with dropout corresponds to sampling from the approximate posterior.

<span style="color: blue;">

$$p(y^*|y,x^*) \approx \frac{1}{S}\sum_{s=1}^S p(y^*|w^{(s)}, x^*), \quad w^{(s)} \sim q_{\text{dropout}}(w)$$

</span>

**Pros:** Zero extra training cost, adds uncertainty estimates to any dropout network. 
 
**Cons:** The theoretical justification is approximate; dropout rate $p$ is a proxy for the prior strength.


##### **12.9 Mean-field VI for BNNs**


Apply the BBVI algorithm (week 11) with a mean-field Gaussian over all weights:

$$q(w) = \prod_i \mathcal{N}(w_i | m_i, v_i)$$

This requires storing $2D$ variational parameters (means + variances for all weights), which is feasible but doubles the parameter count.

The gradient is computed via the reparameterization trick over $S$ weight samples per mini-batch. This is sometimes called **Bayes by Backprop** (Blundell et al., 2015).

**Practical issue:** Mean-field assumes all weights are independent under the posterior. This is a strong assumption so correlations between weights encoding the same computation are discarded.


##### **12.10 Comparison of BNN methods**



<span style="color: blue;">

| Method | Cost | Epistemic Uncertainty | Notes |
|--------|------|-----------------------|-------|
| MAP | $1\times$ train | None (plug-in) | Overconfident OOD |
| Deep Ensemble | $S\times$ train | Yes (via diversity) | Best empirical performance, expensive |
| LLLA | $1\times$ train + cheap post-hoc | Yes (last layer only) | Good trade-off, easy to add to pretrained models |
| MC Dropout | $1\times$ train | Approximate | Free uncertainty, theoretical justification loose |
| Mean-field VI (BBVI) | $1\times$ train (but 2× params) | Yes (mean-field) | Principled, underestimates covariances |
| Full Laplace | $1\times$ train + $\mathcal{O}(D^3)$ | Yes | Often impractical for large networks |
| MCMC (HMC) | Extremely slow | Yes (gold standard) | Used for benchmarking, not production |

</span>
